# CareLoop Clinics — No-Show Guard
## Part 1: Python & Data Foundations: Ingest, Validate, Clean, Enrich
**Author**: Data Analytics Team  
**Deliverable**: Data pipeline from raw export to verified SQL schema and feature table (`appointments_clean.csv`)


### Executive Context & Architecture
CareLoop Clinics operates eight outpatient facilities across six Indian cities. Patient no-shows leave doctor consulting slots permanently idle, depriving other patients of medical care and undermining clinic capacity utilization.

In Part 1, we establish a robust data foundation:
1. **Task 1.1**: Foundations-first ingestion using Python's native `csv` module with strict type casting, logging malformed rows to `rejected_rows.json`.
2. **Task 1.2**: Systematic DataFrame cleaning in pandas: deduplication, text normalization, and date parsing.
3. **Task 1.3**: Handling clinical domain validity, median imputation, and IQR outlier capping.
4. **Task 1.4**: Feature engineering (`lead_days`, `no_show`) and vectorized NumPy descriptive analysis.
5. **Task 1.5**: External clinic enrichment via the Open-Meteo REST API with automated retry handling.
6. **Task 1.6**: Clean export to `appointments_clean.csv` and automated generation of `load_data.sql`.


---
### Task 1.1 — Read the file the foundations way (csv module)
We open `appointments.csv` using Python's built-in `csv.DictReader`. We convert numeric fields (`age`, `fee_inr`, `prior_no_shows` to `int` and `distance_km` to `float`). Rows that trigger a `ValueError` (such as empty strings) are separated into `rejected` and exported to `rejected_rows.json`.


In [1]:
import csv
import json

def parse_row(row):
    r = dict(row)
    r['age'] = int(r['age'])
    r['fee_inr'] = int(r['fee_inr'])
    r['prior_no_shows'] = int(r['prior_no_shows'])
    r['distance_km'] = float(r['distance_km'])
    return r

accepted = []
rejected = []

with open('data/appointments.csv', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames
    print(f"Header: {fieldnames}")
    raw_rows = list(reader)
    print(f"Total raw rows read: {len(raw_rows)}")
    
    for row in raw_rows:
        try:
            accepted.append(parse_row(row))
        except ValueError:
            rejected.append(row)

print(f"Accepted rows (valid types): {len(accepted)}")
print(f"Rejected rows (type conversion failures): {len(rejected)}")

# Save rejected rows to rejected_rows.json
with open('rejected_rows.json', 'w', encoding='utf-8') as f:
    json.dump(rejected, f, indent=2)

print("Saved rejected_rows.json with indent=2.")


Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total raw rows read: 1020
Accepted rows (valid types): 988
Rejected rows (type conversion failures): 32
Saved rejected_rows.json with indent=2.


Notice that all 32 rejected rows failed due to missing numeric values (12 blank `age`, 20 blank `distance_km`). Rows with biologically impossible ages (e.g. `250`) were accepted here because syntactically they are valid integers — a type check is not a domain validity check.


---
### Task 1.2 — Inspect and clean, pass 1 (pandas)
Starting fresh from the original `data/appointments.csv`, we use pandas to inspect nulls, drop exact duplicate records, standardize strings, and parse timestamps.


In [2]:
import pandas as pd
import numpy as np

# Load original export
df = pd.read_csv('data/appointments.csv')

print(f"Initial raw shape: {df.shape}")
print("\nMissing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nDuplicate row count: {df.duplicated().sum()}")

# (a) Drop exact duplicates
df = df.drop_duplicates()
print(f"Row count after dropping exact duplicates: {len(df)}")

# (b) Clean booking_channel
raw_channels = df['booking_channel'].unique()
print(f"Distinct booking_channel spellings before cleaning ({len(raw_channels)}): {sorted(raw_channels)}")
df['booking_channel'] = df['booking_channel'].str.strip().str.title()
cleaned_channels = df['booking_channel'].unique()
print(f"Distinct booking_channel values after cleaning ({len(cleaned_channels)}): {sorted(cleaned_channels)}")

# (c) Convert dates
df['appt_date'] = pd.to_datetime(df['appt_date'], errors='coerce')
df['booked_on'] = pd.to_datetime(df['booked_on'], errors='coerce')

# (d) Drop rows missing appt_date or status
df = df.dropna(subset=['appt_date', 'status'])
print(f"Row count after dropping unreadable appt_date or missing status: {len(df)}")


Initial raw shape: (1020, 14)

Missing values per column:
appt_date        1
age             12
distance_km     20
status           4
dtype: int64

Duplicate row count: 20
Row count after dropping exact duplicates: 1000
Distinct booking_channel spellings before cleaning (16): [' App', ' phone', ' walk-in', 'APP ', 'App', 'App ', 'PHONE', 'Phone', 'Phone ', 'WALK-IN', 'Walk-In', 'Walk-in', 'Walk-in ', 'app', 'phone', 'walk-in']
Distinct booking_channel values after cleaning (3): ['App', 'Phone', 'Walk-In']
Row count after dropping unreadable appt_date or missing status: 990


---
### Task 1.3 — Clean, pass 2: validity, gaps, and outliers
We now enforce clinical validity bounds:
- Ages < 1 or > 100 are treated as entry errors and set to missing (`np.nan`).
- Missing ages are imputed with the median and rounded to integers.
- Missing `distance_km` values are imputed with their median.
- IQR fences (Q1 - 1.5×IQR, Q3 + 1.5×IQR) are computed on filled data and clipped using `np.clip`.


In [3]:
# (a) Domain validity: age between 1 and 100
invalid_age_mask = (df['age'] < 1) | (df['age'] > 100)
invalid_age_count = invalid_age_mask.sum()
df.loc[invalid_age_mask, 'age'] = np.nan
print(f"Invalid age values set to missing: {invalid_age_count}")
print(f"Total missing age values before imputation: {df['age'].isnull().sum()}")

# (b) Impute missing values with medians
median_age = df['age'].median()
median_distance = df['distance_km'].median()
print(f"Median age for imputation: {median_age}")
print(f"Median distance_km for imputation: {median_distance}")

df['age'] = df['age'].fillna(median_age).round().astype(int)
df['distance_km'] = df['distance_km'].fillna(median_distance)

# (c) Outlier capping for distance_km and fee_inr
# Distance IQR
q1_dist = df['distance_km'].quantile(0.25)
q3_dist = df['distance_km'].quantile(0.75)
iqr_dist = q3_dist - q1_dist
lower_dist = q1_dist - 1.5 * iqr_dist
upper_dist = q3_dist + 1.5 * iqr_dist

dist_capped = ((df['distance_km'] > upper_dist) | (df['distance_km'] < lower_dist)).sum()
df['distance_km'] = np.clip(df['distance_km'], lower_dist, upper_dist)

# Fee IQR
q1_fee = df['fee_inr'].quantile(0.25)
q3_fee = df['fee_inr'].quantile(0.75)
iqr_fee = q3_fee - q1_fee
lower_fee = q1_fee - 1.5 * iqr_fee
upper_fee = q3_fee + 1.5 * iqr_fee

fee_capped = ((df['fee_inr'] > upper_fee) | (df['fee_inr'] < lower_fee)).sum()
df['fee_inr'] = np.clip(df['fee_inr'], lower_fee, upper_fee)

# (d) Rounding
df['distance_km'] = df['distance_km'].round(1)
df['fee_inr'] = df['fee_inr'].round().astype(int)

print(f"Distance: Q1={q1_dist}, Q3={q3_dist}, Upper Fence={upper_dist}, Capped={dist_capped}")
print(f"Fee: Q1={q1_fee}, Q3={q3_fee}, Upper Fence={upper_fee}, Capped={fee_capped}")
print(f"Maximum fee after capping: {df['fee_inr'].max()}")


Invalid age values set to missing: 5
Total missing age values before imputation: 17
Median age for imputation: 48.0
Median distance_km for imputation: 8.25
Distance: Q1=4.625, Q3=13.0, Upper Fence=25.5625, Capped=26
Fee: Q1=500.0, Q3=787.5, Upper Fence=1218.75, Capped=3
Maximum fee after capping: 1219


---
### Task 1.4 — Feature Engineering and Vectorized NumPy Summary
We engineer:
- `lead_days`: Calendar days elapsed between booking creation and visit date.
- `no_show`: Binary indicator (1 if status is "No-Show", else 0).
We then inspect the distribution using vectorized NumPy calculations without loops.


In [4]:
# Feature creation
df['lead_days'] = (df['appt_date'] - df['booked_on']).dt.days
df['no_show'] = (df['status'] == 'No-Show').astype(int)

print(f"lead_days range: {df['lead_days'].min()} to {df['lead_days'].max()}")
no_shows_total = df['no_show'].sum()
print(f"No-shows: {no_shows_total} out of {len(df)} ({no_shows_total / len(df) * 100:.1f}%)")

# Extract matrix X for summary
X_num = df[['age', 'distance_km', 'fee_inr', 'lead_days']].to_numpy()

# (a) Column means
col_means = np.mean(X_num, axis=0)
print(f"Column means [age, distance_km, fee_inr, lead_days]: {np.round(col_means, 2)}")

# (b) Mask for lead_days > 14
mask_lead = X_num[:, 3] > 14
print(f"Rows with lead_days > 14: count = {mask_lead.sum()}, share = {mask_lead.mean() * 100:.1f}%")

# (c) Z-scores for distance_km
dist_col = X_num[:, 1]
z_dist = (dist_col - np.mean(dist_col)) / np.std(dist_col)
z_outliers = np.sum(np.abs(z_dist) > 2)
print(f"Rows with |z| > 2 on distance_km: {z_outliers}")


lead_days range: 1 to 45
No-shows: 218 out of 990 (22.0%)
Column means [age, distance_km, fee_inr, lead_days]: [ 48.17   9.5  635.71  11.82]
Rows with lead_days > 14: count = 296, share = 29.9%
Rows with |z| > 2 on distance_km: 53


---
### Task 1.5 — Enrich the clinics from a live API (Open-Meteo)
We fetch baseline temperature readings from Open-Meteo for each of our 8 clinic locations, implementing timeout and retry handling.


In [5]:
import requests

clinics_df = pd.read_csv('data/clinics.csv')
temps = []

for idx, row in clinics_df.iterrows():
    lat, lon, name = row['latitude'], row['longitude'], row['clinic_name']
    url = "https://api.open-meteo.com/v1/forecast"
    params = {"latitude": lat, "longitude": lon, "hourly": "temperature_2m", "forecast_days": 1}
    temp_val = None
    
    for attempt in range(2):
        try:
            resp = requests.get(url, params=params, timeout=10)
            if resp.status_code == 200:
                data = resp.json()
                if "hourly" in data and "temperature_2m" in data["hourly"]:
                    temp_val = data["hourly"]["temperature_2m"][0]
                    break
            else:
                print(f"Attempt {attempt+1} failed for {name}: Status {resp.status_code}")
        except requests.exceptions.RequestException as e:
            if attempt == 1:
                print(f"Clinic {name} failed after retry: {e}")
                
    temps.append(temp_val)
    print(f"{name} ({row['city']}): temp_c = {temp_val}")

clinics_df['temp_c'] = temps
clinics_df.to_csv('clinics_enriched.csv', index=False)
print("Saved clinics_enriched.csv successfully!")


Kothrud Family Clinic (Pune): temp_c = 15.6
Hadapsar Care Centre (Pune): temp_c = 17.6
Sitabuldi Health Point (Nagpur): temp_c = 19.9
Malviya Nagar Clinic (Jaipur): temp_c = 19.7
Vaishali Nagar Clinic (Jaipur): temp_c = 19.7
Edappally Medical Centre (Kochi): temp_c = 25.6
Vijay Nagar Clinic (Indore): temp_c = 19.7
Gomti Nagar Care Point (Lucknow): temp_c = 18.9
Saved clinics_enriched.csv successfully!


---
### Task 1.6 — Export the clean table and write SQL load script
We sort the cleaned dataset by `appointment_id`, format dates as `YYYY-MM-DD`, verify zero missing values, and export `appointments_clean.csv`. We then write `load_data.sql` with clinic inserts first (for foreign key integrity).


In [6]:
# Sort and format dates
df = df.sort_values('appointment_id').reset_index(drop=True)
df['booked_on'] = df['booked_on'].dt.strftime('%Y-%m-%d')
df['appt_date'] = df['appt_date'].dt.strftime('%Y-%m-%d')

cols_order = [
    'appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date',
    'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel',
    'visit_type', 'reminder_sent', 'prior_no_shows', 'status',
    'lead_days', 'no_show'
]
df = df[cols_order]

# Integrity checks
assert df['appointment_id'].nunique() == 990, "appointment_id is not unique"
assert df.isnull().sum().sum() == 0, "Missing values found in cleaned dataset"
print(f"Integrity check passed: {df['appointment_id'].nunique()} unique IDs, 0 nulls across 16 columns.")

# Export clean CSV
df.to_csv('appointments_clean.csv', index=False)
print("Exported appointments_clean.csv (990 rows, 16 columns).")

# Generate load_data.sql
sql_lines = []
clinics_enriched = pd.read_csv('clinics_enriched.csv')

for _, c in clinics_enriched.iterrows():
    temp_val = 'NULL' if pd.isna(c['temp_c']) else c['temp_c']
    sql = f"INSERT INTO clinics VALUES ('{c['clinic_id']}','{c['clinic_name']}','{c['city']}','{c['tier']}',{c['latitude']},{c['longitude']},{temp_val});"
    sql_lines.append(sql)

for _, r in df.iterrows():
    sql = (
        f"INSERT INTO appointments VALUES ("
        f"'{r['appointment_id']}','{r['patient_id']}','{r['clinic_id']}',"
        f"'{r['booked_on']}','{r['appt_date']}',{r['age']},'{r['gender']}',"
        f"{r['distance_km']:.1f},{r['fee_inr']},'{r['booking_channel']}',"
        f"'{r['visit_type']}','{r['reminder_sent']}',{r['prior_no_shows']},"
        f"'{r['status']}',{r['lead_days']},{r['no_show']});"
    )
    sql_lines.append(sql)

with open('load_data.sql', 'w', encoding='utf-8') as f:
    f.write('\n'.join(sql_lines) + '\n')

print(f"Generated load_data.sql with {len(sql_lines)} INSERT statements (8 clinics + 990 appointments).")


Integrity check passed: 990 unique IDs, 0 nulls across 16 columns.
Exported appointments_clean.csv (990 rows, 16 columns).
Generated load_data.sql with 998 INSERT statements (8 clinics + 990 appointments).
